# Gold layer definitions

Dimensional model (star schema) built from the Silver layer, plus the pre-aggregated
tables used by the dashboard. Like `layers`, this notebook only defines things, it does
not write any table. It is loaded with `%run ./gold_layer` by `pipeline_utils`, where
the Gold tables are created and loaded.

| Table | Type | Grain |
|---|---|---|
| `gold_dim_country` | Dimension | one row per country / aggregate |
| `gold_dim_indicator` | Dimension | one row per indicator |
| `gold_dim_year` | Dimension | one row per year |
| `gold_fact_indicator_values` | Fact | country x indicator x year |
| `gold_agg_indicator_trends` | Aggregate | group (region / income level / world) x indicator x year |
| `gold_agg_yoy_growth` | Aggregate | country x indicator x reported year |
| `gold_agg_latest_values` | Aggregate | country x indicator (latest reported year) |

In [0]:
import os
import json

# Reference data: the World Bank country list (/v2/country?per_page=400), saved once by
# the extraction notebook. The indicator API does not say if an entity is a country or
# an aggregate, or which region / income group it belongs to, so this file is used for that.
# The path is relative to this folder, like the raw files in DEFAULT_SOURCES.
COUNTRY_METADATA_FILE = "../data/raw/Metadata/countries_metadata.json"

# Entities that appear in the indicator data but are not in the country list.
# Silver country_key -> (entity_type, region)
EXTRA_ENTITIES = {
    # LAC Equity Lab sub-regions and other groupings
    "ANR": ("Aggregate", None),   # Andean Region
    "LCR": ("Aggregate", None),   # Latin America and the Caribbean
    "MCA": ("Aggregate", None),   # Central America
    "SCE": ("Aggregate", None),   # Southern Cone
    "F1": ("Aggregate", None),    # Fragile and conflict affected situations
    "NAME_GLOBAL_PARTNERSHIP_FOR_EDUCATION": ("Aggregate", None),
    "NAME_LENDING_CATEGORY_NOT_CLASSIFIED": ("Aggregate", None),
    # Territories sent with an empty country id
    "NAME_ANGUILLA": ("Country", "Latin America & Caribbean"),
    "NAME_MONTSERRAT": ("Country", "Latin America & Caribbean"),
    "NAME_NETHERLANDS_ANTILLES": ("Country", "Latin America & Caribbean"),
    "NAME_COOK_ISLANDS": ("Country", "East Asia & Pacific"),
    "NAME_NIUE": ("Country", "East Asia & Pacific"),
    "NAME_TOKELAU": ("Country", "East Asia & Pacific"),
    "NAME_MAYOTTE": ("Country", "Sub-Saharan Africa"),
}

# Units where adding the values of several countries gives a meaningful total
# (per capita indicators are excluded by name, see build_dim_indicator)
ADDITIVE_UNITS = ["sq. km", "hectares", "current US$", "BoP, current US$", "current $", "constant 2005 $", "people"]

# Indicators whose name has no unit in brackets
UNIT_OVERRIDES = {
    "SP.POP.TOTL": "people",
    "EN.GHG.CO2.PC.CE.AR5": "t CO2e per capita",
    "SI.POV.DDAY": "% of population",
}

# Column order of the Gold tables (same order as the DDL in pipeline_utils)
DIM_COUNTRY_ATTRS = ["country_name", "iso3_code", "entity_type", "region", "income_level",
                     "lending_type", "capital_city", "latitude", "longitude", "source_country_keys"]
DIM_INDICATOR_ATTRS = ["indicator_name", "category", "source_dataset", "unit", "is_additive"]
DIM_YEAR_ATTRS = ["decade", "decade_label", "five_year_period"]

FACT_KEY = ["country_sk", "indicator_sk", "year_key"]
FACT_COLUMNS = FACT_KEY + ["value", "is_reported", "decimal_places", "source_batch_id",
                           "silver_load_timestamp", "load_timestamp"]
TRENDS_COLUMNS = ["group_type", "group_name", "indicator_sk", "year_key", "country_count",
                  "reporting_countries", "coverage_pct", "avg_value", "median_value",
                  "min_value", "max_value", "sum_value", "load_timestamp"]
YOY_COLUMNS = FACT_KEY + ["value", "prev_year_key", "prev_value", "years_gap", "abs_change",
                          "pct_change", "annualized_growth_pct", "load_timestamp"]
LATEST_COLUMNS = ["country_sk", "indicator_sk", "latest_year_key", "latest_value", "first_year_key",
                  "reported_years", "load_timestamp"]

In [0]:
def _text(v):
    v = (v or "").strip()
    return v or None


def _number(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


REFERENCE_SCHEMA = StructType([
    StructField("ref_code", StringType(), False),
    StructField("ref_iso3", StringType(), True),
    StructField("ref_name", StringType(), True),
    StructField("ref_entity_type", StringType(), True),
    StructField("ref_region", StringType(), True),
    StructField("ref_income_level", StringType(), True),
    StructField("ref_lending_type", StringType(), True),
    StructField("ref_capital_city", StringType(), True),
    StructField("ref_latitude", DoubleType(), True),
    StructField("ref_longitude", DoubleType(), True),
])

ALIAS_SCHEMA = StructType([
    StructField("alias_key", StringType(), False),
    StructField("alias_code", StringType(), False),
])


def read_country_reference(data_dir):
    """Read countries_metadata.json and return two small DataFrames.

    reference: one row per entity, keyed by the World Bank 2-letter id (ref_code).
    aliases:   every key an entity can have in Silver -> its ref_code. The LAC Equity Lab
               indicators use ISO3 ids (ARG, BRA, ...) while all other indicators use the
               2-letter id (AR, BR, ...), so both have to point to the same country.
    """
    with open(os.path.join(data_dir, COUNTRY_METADATA_FILE)) as f:
        countries = json.load(f)

    reference, aliases = [], {}
    for c in countries:
        code = c["iso2Code"].strip()
        is_aggregate = _text(c["region"]["value"]) == "Aggregates"
        reference.append((
            code, _text(c["id"]), _text(c["name"]),
            "Aggregate" if is_aggregate else "Country",
            None if is_aggregate else _text(c["region"]["value"]),
            None if is_aggregate else _text(c["incomeLevel"]["value"]),
            None if is_aggregate else _text(c["lendingType"]["value"]),
            _text(c["capitalCity"]), _number(c["latitude"]), _number(c["longitude"]),
        ))
        aliases.setdefault(code, code)
        aliases.setdefault(c["id"].strip(), code)

    for key, (entity_type, region) in EXTRA_ENTITIES.items():
        reference.append((key, None, None, entity_type, region, None, None, None, None, None))
        aliases.setdefault(key, key)

    return (spark.createDataFrame(reference, REFERENCE_SCHEMA),
            spark.createDataFrame(list(aliases.items()), ALIAS_SCHEMA))


def conform_country(df, aliases):
    """Add dim_country_code: the business key of the row in gold_dim_country.
    Keys that are not in the reference keep their Silver country_key."""
    return (
        df.join(F.broadcast(aliases), df["country_key"] == aliases["alias_key"], "left")
        .withColumn("dim_country_code", F.coalesce(F.col("alias_code"), F.col("country_key")))
        .drop("alias_key", "alias_code")
    )

In [0]:
def build_dim_country(silver, reference, aliases, load_ts):
    """One row per country or aggregate. A country sent under two ids (AR / ARG) is one row.
    silver_rows is only used for logging and is not written to the table."""
    entities = (
        conform_country(silver.select("country_key", "country_name", "country_iso3"), aliases)
        .groupBy("dim_country_code")
        .agg(
            F.count(F.lit(1)).alias("silver_rows"),
            F.max("country_name").alias("silver_name"),
            F.max("country_iso3").alias("silver_iso3"),
            F.concat_ws(", ", F.sort_array(F.collect_set("country_key"))).alias("source_country_keys"),
        )
    )
    return (
        entities.join(F.broadcast(reference), entities["dim_country_code"] == reference["ref_code"], "left")
        .select(
            F.col("dim_country_code").alias("country_code"),
            F.coalesce("ref_name", "silver_name").alias("country_name"),
            F.coalesce("ref_iso3", "silver_iso3").alias("iso3_code"),
            F.coalesce(F.col("ref_entity_type"), F.lit("Unclassified")).alias("entity_type"),
            F.col("ref_region").alias("region"),
            F.col("ref_income_level").alias("income_level"),
            F.col("ref_lending_type").alias("lending_type"),
            F.col("ref_capital_city").alias("capital_city"),
            F.col("ref_latitude").alias("latitude"),
            F.col("ref_longitude").alias("longitude"),
            "source_country_keys",
            F.lit(load_ts).cast(TimestampType()).alias("load_timestamp"),
            "silver_rows",
        )
    )


def indicator_category(c):
    # Theme from the code prefix (World Bank codes start with the topic, e.g. NY = national accounts)
    return (
        F.when(c.startswith("NY.") | c.startswith("FP.") | c.startswith("5.") | c.startswith("6."), "Economy")
        .when(c.startswith("SP.POP"), "Population")
        .when(c.startswith("SH.") | c.startswith("SP.DYN"), "Health")
        .when(c.startswith("SE.") | c.startswith("BAR.") | c.startswith("2."), "Education")
        .when(c.startswith("SL.") | c.startswith("9."), "Labor")
        .when(c.startswith("SI."), "Poverty")
        .when(c.startswith("IT."), "Connectivity")
        .when(c.startswith("EN."), "Environment")
        .when(c.startswith("EG."), "Energy")
        .when(c.startswith("AG."), "Agriculture & Land")
        .when(c.startswith("BX.") | c.startswith("DC."), "Aid & Grants")
        .otherwise("Other")
    )


def indicator_source(c):
    return (
        F.when(c.startswith("BAR."), "Barro-Lee Educational Attainment")
        .when(c.rlike(r"^[0-9]"), "LAC Equity Lab")
        .otherwise("World Development Indicators")
    )


def indicator_unit(code, name):
    # The unit is the last part in brackets, e.g. "Forest area (% of land area)"
    raw = F.regexp_extract(name, r"\(([^()]*)\)[^()]*$", 1)
    override = F.lit(None).cast(StringType())
    for indicator, unit in UNIT_OVERRIDES.items():
        override = F.when(code == indicator, F.lit(unit)).otherwise(override)
    return F.coalesce(
        override,
        F.when(raw.rlike(r"^[0-9]{4}-[0-9]{4}$"), F.lit("% per year"))  # "Annualized Mean Income Growth (2004-2014)"
        .when(raw != "", raw)
        .when(name.contains("Percentage"), F.lit("%")),                 # Barro-Lee
    )


def build_dim_indicator(silver, load_ts):
    per_capita = F.lower(F.col("indicator_name")).contains("per capita")
    return (
        silver.groupBy("indicator_code")
        .agg(F.count(F.lit(1)).alias("silver_rows"), F.max("indicator_name").alias("indicator_name"))
        .withColumn("unit", indicator_unit(F.col("indicator_code"), F.col("indicator_name")))
        .select(
            "indicator_code",
            "indicator_name",
            indicator_category(F.col("indicator_code")).alias("category"),
            indicator_source(F.col("indicator_code")).alias("source_dataset"),
            "unit",
            F.coalesce(F.col("unit").isin(ADDITIVE_UNITS) & ~per_capita, F.lit(False)).alias("is_additive"),
            F.lit(load_ts).cast(TimestampType()).alias("load_timestamp"),
            "silver_rows",
        )
    )


def build_dim_year(first_year, last_year, load_ts):
    """Every year from the first to the last year in Silver (also years without data)."""
    period_start = F.col("year_key") - F.col("year_key") % 5
    return (
        spark.range(first_year, last_year + 1)
        .select(F.col("id").cast(IntegerType()).alias("year_key"))
        .withColumn("decade", (F.col("year_key") - F.col("year_key") % 10).cast(IntegerType()))
        .withColumn("decade_label", F.concat(F.col("decade").cast(StringType()), F.lit("s")))
        .withColumn("five_year_period", F.concat_ws("-", period_start.cast(StringType()), (period_start + 4).cast(StringType())))
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
    )

In [0]:
def map_fact_keys(silver_rows, aliases, dim_country, dim_indicator):
    """Replace the Silver business keys with the Gold surrogate keys.

    Left joins, so a row without a dimension member is kept with a NULL key. The loader
    checks for that and fails the step instead of silently dropping rows.
    """
    countries = dim_country.select("country_sk", F.col("country_code").alias("dim_country_code"))
    indicators = dim_indicator.select("indicator_sk", F.col("indicator_code").alias("_indicator_code"))
    return (
        conform_country(silver_rows, aliases)
        .join(F.broadcast(countries), "dim_country_code", "left")
        .join(F.broadcast(indicators), F.col("indicator_code") == F.col("_indicator_code"), "left")
        .drop("_indicator_code")
        .withColumn("year_key", F.col("year"))
    )


def fact_source(mapped, load_ts):
    """One row per fact key. AR and ARG never share an indicator in this data, but if they
    ever did, the most recently loaded row wins (and the 2-letter id before the ISO3 id)."""
    w = Window.partitionBy(*FACT_KEY).orderBy(
        F.col("load_timestamp").desc(),
        (F.col("country_key") == F.col("dim_country_code")).desc(),
    )
    return (
        mapped.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1)
        .select(
            *FACT_KEY, "value", "is_reported", "decimal_places", "source_batch_id",
            F.col("load_timestamp").alias("silver_load_timestamp"),
            F.lit(load_ts).cast(TimestampType()).alias("load_timestamp"),
        )
    )

In [0]:
def build_agg_trends(fact, dim_country, dim_indicator, load_ts):
    """Indicator values summarised per region, per income level and for the whole world.

    Only real countries are used. The World Bank aggregates (World, South Asia, High income,
    ...) are left out, otherwise their values would be counted together with the countries.
    avg_value is a simple average across countries (not population weighted), and
    sum_value is only filled for indicators with an additive unit (sq. km, US$, ...).
    """
    base = (
        fact.join(F.broadcast(dim_country.filter(F.col("entity_type") == "Country")
                              .select("country_sk", "region", "income_level")), "country_sk")
        .join(F.broadcast(dim_indicator.select("indicator_sk", "is_additive")), "indicator_sk")
    )

    def summarise(group_type, group_name):
        return (
            base.filter(group_name.isNotNull())
            .groupBy(F.lit(group_type).alias("group_type"), group_name.alias("group_name"),
                     "indicator_sk", "year_key")
            .agg(
                F.count(F.lit(1)).alias("country_count"),
                F.count("value").alias("reporting_countries"),
                F.avg("value").alias("avg_value"),
                F.percentile_approx("value", 0.5).alias("median_value"),
                F.min("value").alias("min_value"),
                F.max("value").alias("max_value"),
                F.when(F.max("is_additive"), F.sum("value")).alias("sum_value"),
            )
        )

    return (
        summarise("Region", F.col("region"))
        .unionByName(summarise("Income level", F.col("income_level")))
        .unionByName(summarise("World", F.lit("World")))
        .withColumn("coverage_pct", F.round(F.col("reporting_countries") * 100 / F.col("country_count"), 2))
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
        .select(*TRENDS_COLUMNS)
    )


def build_agg_yoy(fact, load_ts):
    """Change of every reported value against the previous reported year of the same
    country and indicator. Many indicators are not published every year (Barro-Lee only
    every 5 years), so years_gap is kept and annualized_growth_pct spreads the change
    evenly over the gap."""
    w = Window.partitionBy("country_sk", "indicator_sk").orderBy("year_key")
    return (
        fact.filter(F.col("is_reported"))
        .select(*FACT_KEY, "value")
        .withColumn("prev_year_key", F.lag("year_key").over(w))
        .withColumn("prev_value", F.lag("value").over(w))
        .withColumn("years_gap", F.col("year_key") - F.col("prev_year_key"))
        .withColumn("abs_change", F.col("value") - F.col("prev_value"))
        .withColumn("pct_change", F.when(F.col("prev_value") != 0,
                                         F.col("abs_change") / F.abs(F.col("prev_value")) * 100))
        .withColumn("annualized_growth_pct", F.when(
            (F.col("value") > 0) & (F.col("prev_value") > 0),
            (F.pow(F.col("value") / F.col("prev_value"), 1 / F.col("years_gap")) - 1) * 100))
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
        .select(*YOY_COLUMNS)
    )


def build_agg_latest(fact, load_ts):
    """Latest reported value of every country and indicator. Survey based indicators such
    as literacy or poverty are only reported in some years, so a map or a ranking should
    use the latest available value instead of one fixed year."""
    return (
        fact.filter(F.col("is_reported"))
        .groupBy("country_sk", "indicator_sk")
        .agg(
            F.max("year_key").alias("latest_year_key"),
            F.max_by("value", "year_key").alias("latest_value"),
            F.min("year_key").alias("first_year_key"),
            F.count(F.lit(1)).alias("reported_years"),
        )
        .withColumn("load_timestamp", F.lit(load_ts).cast(TimestampType()))
        .select(*LATEST_COLUMNS)
    )